# Document Loading & Chunking

En este cuaderno cargaremos documentos de diferentes fuentes de datos, haremos el troceado de estos y los guardaremos en una vectorstore para finalmente hacer consultas.


## Install dependencies

In [1]:
!pip install "langchain>=0.3" "langchain-community>=0.3" "langchain-openai>=0.2" pypdf pytubefix youtube-transcript-api openai langchain_experimental langchain_openai

## Document Loading

Cargaremos 3 tipos de documentos. Un PDF de un sitio de internet, la transcripción de un video de Youtube y el contenido de una página web.







### PDF Loading

Cargaremos la transcripción de un curso de Andrew Ng: CS229 course

[Stanford CS229: Machine Learning Course, Lecture 1 - Andrew Ng (Autumn 2018)](https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf)

In [2]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf")
lecture1_pages = loader.load()

In [3]:
len(lecture1_pages)

22

In [4]:
page = lecture1_pages[15]

In [5]:
print(page.page_content[0:500])

classes teach. And this is something I'm really convinced is a huge deal, and so by the 
end of this class, I hope all of you will be master carpenters. I hope all of you will be 
really good at applying these learning algorithms and getting them to work amazingly 
well in many problems. Okay?  
Let's see. So [inaudible] the board. After learning theory, there's another class of learning 
algorithms that I then want to teach you about, and that's unsupervised learning. So you 
recall, right, a l


In [6]:
page.metadata

{'producer': 'Acrobat Distiller 8.1.0 (Windows)',
 'creator': 'PScript5.dll Version 5.2.2',
 'creationdate': '2008-07-11T11:25:23-07:00',
 'author': '',
 'moddate': '2008-07-11T11:25:23-07:00',
 'title': '',
 'source': 'https://see.stanford.edu/materials/aimlcs229/transcripts/MachineLearning-Lecture01.pdf',
 'total_pages': 22,
 'page': 15,
 'page_label': '16'}

### Youtube Loading

Cargaremos la transcripción de un video de Youtube.


El video => [Stanford CS229: Machine Learning - Linear Regression and Gradient Descent | Lecture 2 (Autumn 2018)](https://www.youtube.com/watch?v=4b4MUYve_U8)

In [7]:
from pytubefix import YouTube

In [8]:
from langchain_community.document_loaders import YoutubeLoader

loader = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=4b4MUYve_U8", add_video_info=False
)

lecture2_pages = loader.load()

In [9]:
lecture2_pages[0].page_content[0:500]

"Morning and welcome back. So what we'll see today in class is the first in-depth discussion of a learning algorithm, linear regression, and in particular, over the next, what, hour and a bit you'll see linear regression, batch and stochastic gradient descent is an algorithm for fitting linear regression models, and then the normal equations, um, uh, as a way of- as a very efficient way to let you fit linear models. Um, and we're going to define notation, and a few concepts today that will lay th"

Por si la API de YT está bloqueada

> ⚠️ Una celda con un transcript hardcodeado de ~60KB que la 1ª edición usaba como fallback para `pytube` (ahora roto) ha sido eliminada. Con `pytubefix` el fallback no es necesario.

### Website Loading

En este caso cargaremos un Markdown de Github de la biblioteca Pytorch. Su readme para poder extraer información: [Readme.md](https://github.com/pytorch/pytorch/blob/main/README.md)

In [10]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://raw.githubusercontent.com/pytorch/pytorch/main/README.md")
#loader = WebBaseLoader("https://www.codemancers.pro/")

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [11]:
lecture3_pages = loader.load()

In [12]:
print(lecture3_pages[0].page_content[:500])







--------------------------------------------------------------------------------

PyTorch is a Python package that provides two high-level features:
- Tensor computation (like NumPy) with strong GPU acceleration
- Deep neural networks built on a tape-based autograd system

You can reuse your favorite Python packages such as NumPy, SciPy, and Cython to extend PyTorch when needed.

Our trunk health (Continuous Integration signals) can be found at [hud.pytorch.org](https://hud.pytorch.org/ci/


## Document Chunking

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter


Configuraremos un tamaño de Chunk de 26, con un solapamiento de 4


### Fixed-size

In [14]:
chunk_size = 26
chunk_overlap = 4

In [15]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)
c_splitter = CharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

In [16]:
text1 = 'abcdefghijklmnopqrstuvwxyz'
print(r_splitter.split_text(text1))
print(c_splitter.split_text(text1))

['abcdefghijklmnopqrstuvwxyz']
['abcdefghijklmnopqrstuvwxyz']


In [17]:
text2 = 'abcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyzabcdefghijklmnopqrstuvwxyz'
r_splitter.split_text(text2)

['abcdefghijklmnopqrstuvwxyz',
 'wxyzabcdefghijklmnopqrstuv',
 'stuvwxyzabcdefghijklmnopqr',
 'opqrstuvwxyzabcdefghijklmn',
 'klmnopqrstuvwxyzabcdefghij',
 'ghijklmnopqrstuvwxyz']

### Recursive Structure Aware Splitting

In [18]:
chapter2_lotr = """2. Concerning Pipe-weed

There is another astonishing thing about Hobbits of old that must be
mentioned, an astonishing habit: they imbibed or inhaled, through pipes of
clay or wood, the smoke of the burning leaves of a herb, which they called
pipe-weed or leaf, a variety probably of Nicotiana. A great deal of mystery
surrounds the origin of this peculiar custom, or 'art' as the Hobbits
preferred to call it. All that could be discovered about it in antiquity was
put together by Meriadoc Brandybuck (later Master of Buckland), and since he
and the tobacco of the Southfarthing play a part in the history that
follows, his remarks in the introduction to his Herblore of the Shire may be
quoted.

’This,' he says, 'is the one art that we can certainly claim to be our
own invention. When Hobbits first began to smoke is not known, all the
legends and family histories take it for granted; for ages folk in the Shire
smoked various herbs, some fouler, some sweeter. But all accounts agree that
Tobold Hornblower of Longbottom in the Southfarthing first grew the true
pipe-weed in his gardens in the days of Isengrim the Second, about the year
1070 of Shire-reckoning. The best home-grown still comes from that district,
especially the varieties now known as Longbottom Leaf, Old Toby, and
Southern Star.

'How Old Toby came by the plant is not recorded, for to his dying day
he would not tell. He knew much about herbs, but he was no traveller. It is
said that in his youth he went often to Bree, though he certainly never went
further from the Shire than that. It is thus quite possible that he learned
of this plant in Bree, where now, at any rate, it grows well on the south
slopes of the hill. The Bree-hobbits claim to have been the first actual
smokers of the pipe-weed. They claim, of course, to have done everything
before the people of the Shire, whom they refer to as "colonists"; but in
this case their claim is, I think, likely to be true. And certainly it was
from Bree that the art of smoking the genuine weed spread in the recent
centuries among Dwarves and such other folk, Rangers, Wizards, or wanderers,
as still passed to and fro through that ancient road-meeting. The home and
centre of the an is thus to be found in the old inn of Bree, The Prancing


Pony, that has been kept by the family of Butterbur from time beyond record.

'All the same, observations that I have made on my own many journeys
south have convinced me that the weed itself is not native to our parts of
the world, but came northward from the lower Anduin, whither it was, I
suspect, originally brought over Sea by the Men of Westernesse. It grows
abundantly in Gondor, and there is richer and larger than in the North,
where it is never found wild, and flourishes only in warm sheltered places
like Longbottom. The Men of Gondor call it sweet galenas, and esteem it only
for the fragrance of its flowers. From that land it must have been carried
up the Greenway during the long centuries between the coming of Elendil and
our own day. But even the D®nedain of Gondor allow us this credit: Hobbits
first put it into pipes. Not even the Wizards first thought of that before
we did. Though one Wizard that I knew took up the art long ago, and became
as skilful in it as in all other things that he put his mind to.' """

In [19]:
len(chapter2_lotr)

3306

In [20]:
c_splitter = CharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separator = ' '
)
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=0,
    separators=[".", ",", ":"]
)

In [21]:
c_splitter.split_text(chapter2_lotr)

['2. Concerning Pipe-weed\n\nThere is another astonishing thing about Hobbits of old that must',
 'be\nmentioned, an astonishing habit: they imbibed or inhaled, through pipes of\nclay or wood, the',
 'smoke of the burning leaves of a herb, which they called\npipe-weed or leaf, a variety probably of',
 "Nicotiana. A great deal of mystery\nsurrounds the origin of this peculiar custom, or 'art' as the",
 'Hobbits\npreferred to call it. All that could be discovered about it in antiquity was\nput together by',
 'Meriadoc Brandybuck (later Master of Buckland), and since he\nand the tobacco of the Southfarthing',
 'play a part in the history that\nfollows, his remarks in the introduction to his Herblore of the',
 "Shire may be\nquoted.\n\n’This,' he says, 'is the one art that we can certainly claim to be our\nown",
 'invention. When Hobbits first began to smoke is not known, all the\nlegends and family histories take',
 'it for granted; for ages folk in the Shire\nsmoked various herbs, some f

In [22]:
r_splitter.split_text(chapter2_lotr)

['2',
 '. Concerning Pipe-weed\n\nThere is another astonishing thing about Hobbits of old that must be\nmentioned',
 ', an astonishing habit: they imbibed or inhaled, through pipes of\nclay or wood',
 ', the smoke of the burning leaves of a herb, which they called\npipe-weed or leaf',
 ', a variety probably of Nicotiana',
 '. A great deal of mystery\nsurrounds the origin of this peculiar custom',
 ", or 'art' as the Hobbits\npreferred to call it",
 '. All that could be discovered about it in antiquity was\nput together by Meriadoc Brandybuck (later Master of Buckland)',
 ', and since he\nand the tobacco of the Southfarthing play a part in the history that\nfollows',
 ', his remarks in the introduction to his Herblore of the Shire may be\nquoted',
 ".\n\n’This,' he says, 'is the one art that we can certainly claim to be our\nown invention",
 '. When Hobbits first began to smoke is not known',
 ', all the\nlegends and family histories take it for granted; for ages folk in the Shire\nsmok

In [23]:
text_splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len
)

In [24]:
docs = text_splitter.split_documents(lecture1_pages)

In [25]:
len(docs)

78

In [26]:
len(lecture1_pages)

22

### Content-Aware Splitting


In [27]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

In [28]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

In [29]:
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)
md_header_splits = markdown_splitter.split_text(lecture3_pages[0].page_content)

In [30]:
md_header_splits[0]

Document(metadata={}, page_content='--------------------------------------------------------------------------------  \nPyTorch is a Python package that provides two high-level features:\n- Tensor computation (like NumPy) with strong GPU acceleration\n- Deep neural networks built on a tape-based autograd system  \nYou can reuse your favorite Python packages such as NumPy, SciPy, and Cython to extend PyTorch when needed.  \nOur trunk health (Continuous Integration signals) can be found at [hud.pytorch.org](https://hud.pytorch.org/ci/pytorch/pytorch/main).  \n- [More About PyTorch](#more-about-pytorch)\n- [A GPU-Ready Tensor Library](#a-gpu-ready-tensor-library)\n- [Dynamic Neural Networks: Tape-Based Autograd](#dynamic-neural-networks-tape-based-autograd)\n- [Python First](#python-first)\n- [Imperative Experiences](#imperative-experiences)\n- [Fast and Lean](#fast-and-lean)\n- [Extensions Without Pain](#extensions-without-pain)\n- [Installation](#installation)\n- [Binaries](#binaries)\n

In [31]:
md_header_splits[1]

Document(metadata={'Header 2': 'More About PyTorch'}, page_content='[Learn the basics of PyTorch](https://pytorch.org/tutorials/beginner/basics/intro.html)  \nAt a granular level, PyTorch is a library that consists of the following components:  \n| Component | Description |\n| ---- | --- |\n| [**torch**](https://pytorch.org/docs/stable/torch.html) | A Tensor library like NumPy, with strong GPU support |\n| [**torch.autograd**](https://pytorch.org/docs/stable/autograd.html) | A tape-based automatic differentiation library that supports all differentiable Tensor operations in torch |\n| [**torch.nn**](https://pytorch.org/docs/stable/nn.html) | A neural networks library deeply integrated with autograd designed for maximum flexibility |\n| [**torch.multiprocessing**](https://pytorch.org/docs/stable/multiprocessing.html) | Python multiprocessing, but with magical memory sharing of torch Tensors across processes. Useful for data loading and Hogwild training |\n| [**torch.utils**](https://pyt

### NLP Chunking

Necesitamos un modelo de embedding para hacer el Splitting. En este ejemplo usaremos OpenAI.


In [32]:
import os
import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

¡Ahora implementamos el `SemanticChunker`!

Hoy utilizaremos el umbral `percentile` como ejemplo - pero hay tres estrategias diferentes que podríais utilizar (descripciones tomadas de la documentación del Fragmentador Semántico; el código fuente está en [langchain-experimental](https://github.com/langchain-ai/langchain-experimental/blob/main/libs/experimental/langchain_experimental/text_splitter.py)):

- `percentile` (por defecto) - En este método, se calculan todas las diferencias entre frases y luego cualquier diferencia mayor que el percentil X se divide.

- `standard_deviation` - En este método, cualquier diferencia mayor que X desviaciones estándar se divide.

- `interquartile` - En este método, la distancia intercuartílica se utiliza para dividir los fragmentos.

La idea básica es la siguiente:

1. Divide nuestro documento en frases (basado en `.`, `?`, y `!`)
2. Indexa cada frase basándose en la posición
3. Añade un `buffer_size` (`int`) de frases a cada lado de nuestra frase seleccionada
4. Calcula distancias entre grupos de frases
5. Fusiona grupos basándose en la similitud según los umbrales anteriores

> NOTA: `SemanticChunker` vive en `langchain-experimental`, que en 2026 está en retirada (sunset: funciona, pero ya no recibe mantenimiento y al importarlo verás un `DeprecationWarning`). La idea es lo que importa: si mañana desaparece, es fácil de reimplementar con cualquier modelo de embeddings.


In [33]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

semantic_chunker = SemanticChunker(OpenAIEmbeddings(model="text-embedding-3-small"), breakpoint_threshold_type="percentile")

In [34]:
loader = PyPDFLoader("https://arxiv.org/pdf/2504.21801v1")
lecture1_pages = loader.load()

In [35]:
content = []

for page in lecture1_pages:
  # pypdf a veces extrae medio carácter matemático (un "surrogate" suelto) que rompe el JSON: lo descartamos
  content.append(page.page_content.encode("utf-8", "ignore").decode("utf-8"))

semantic_chunks = semantic_chunker.create_documents(content)

In [36]:
len(semantic_chunks)

82

In [37]:
print(semantic_chunks[0])

page_content='DeepSeek-Prover-V2: Advancing Formal Mathematical Reasoning via
Reinforcement Learning for Subgoal Decomposition
Z.Z. Ren*, Zhihong Shao*, Junxiao Song*, Huajian Xin†, Haocheng Wang†, Wanjia Zhao†, Liyue Zhang, Zhe Fu
Qihao Zhu, Dejian Yang, Z.F. Wu, Zhibin Gou, Shirong Ma, Hongxuan Tang, Yuxuan Liu, Wenjun Gao
Daya Guo, Chong Ruan
DeepSeek-AI
https://github.com/deepseek-ai/DeepSeek-Prover-V2
Abstract
We introduce DeepSeek-Prover-V2, an open-source large language model designed for formal
theorem proving in Lean 4, with initialization data collected through a recursive theorem prov-
ing pipeline powered by DeepSeek-V3. The cold-start training procedure begins by prompting
DeepSeek-V3 to decompose complex problems into a series of subgoals.'


In [38]:
print(semantic_chunks[1])

page_content='The proofs of resolved
subgoals are synthesized into a chain-of-thought process, combined with DeepSeek-V3’s step-
by-step reasoning, to create an initial cold start for reinforcement learning. This process enables
us to integrate both informal and formal mathematical reasoning into a unified model. The
resulting model, DeepSeek-Prover-V2-671B, achieves state-of-the-art performance in neural
theorem proving, reaching 88.9% pass ratio on the MiniF2F-test and solving 49 out of 658
problems from PutnamBench. In addition to standard benchmarks, we introduce ProverBench,
a collection of 325 formalized problems, to enrich our evaluation, including 15 selected prob-
lems from the recent AIME competitions (years 24-25). Further evaluation on these 15 AIME
problems shows that the model successfully solves 6 of them. In comparison, DeepSeek-V3
solves 8 of these problems using majority voting, highlighting that the gap between formal and
informal mathematical reasoning in large lang

In [39]:
print(semantic_chunks[2])

page_content='1. Introduction
The emergence of reasoning capabilities in large language models (LLMs) has revolutionized
numerous areas of artificial intelligence, particularly in the domain of mathematical problem
solving (DeepSeek-AI, 2025). These advancements are largely enabled by the paradigm of
inference-time scaling, most notably through natural language chain-of-thought reasoning
(Jaech et al., 2024). Rather than relying solely on a single forward pass to arrive at an answer,
LLMs can reflect on intermediate reasoning steps, improving both accuracy and interpretability. Despite the success of natural language reasoning in solving competition-level mathematical
problems, its application to formal theorem proving remains fundamentally challenging. LLMs
perform natural language reasoning in an inherently informal manner, relying on heuristics, ap-
proximations, and data-driven guessing patterns that often lack the rigorous structure required
by formal verification systems. In cont

In [40]:
print(semantic_chunks[3])

page_content='These systems
permit no ambiguity, implicit assumptions, or omission of details. Bridging the gap between
informal, high-level reasoning and the syntactic rigor of formal verification systems remains a
longstanding research challenge in neural theorem proving (Yang et al., 2024). To harness the strengths of informal mathematical reasoning in support of formal theorem
proving, a classical approach is to hierarchically decompose formal proofs based on the guidance
of natural-language proof sketches. Jiang et al. (2023) proposed a framework, calledDraft, Sketch,
and Prove (DSP), that leverages a large language model to generate proof sketches in natural
language, which are subsequently translated into formal proof steps. This informal-to-formal
theorem proving paradigm closely mirrors the concept of subgoals in hierarchical reinforcement
learning (Barto and Mahadevan, 2003; Nachum et al., 2018; Eppe et al., 2022), where complex
tasks are broken down into a hierarchy of simpl

## Contextual retrieval · patrón Anthropic 2024

El problema clásico del chunking: un chunk dice "el modelo logró 92% de accuracy" pero **no dice de qué modelo habla**. El embedding lo captura mal y el retrieval falla.

**Solución de Anthropic (sept 2024)**: antes de embeber, prepende a cada chunk un resumen de 50-100 tokens explicando dónde encaja el chunk en el documento completo.

Reportado por Anthropic: **un 35% menos de fallos de retrieval** (top-20) solo con embeddings contextuales, un 49% combinándolo con BM25 contextual y un 67% añadiendo reranking. La carga es generar el contexto con un LLM pequeño (cacheable) una sola vez por chunk, en indexación.

Pseudocódigo:

```python
for chunk in chunks:
    context = llm.invoke(
        f"Resume en una frase dónde encaja este chunk en el documento.\n"
        f"Documento entero (cacheado): {full_doc}\n"
        f"Chunk: {chunk.page_content}"
    )
    chunk.page_content = f"{context}\n\n{chunk.page_content}"
    # ahora embebes este chunk enriquecido
```

**Clave**: el documento entero va dentro de un bloque cacheable (Anthropic prompt caching) → la 2ª llamada solo paga 0.1× del precio. Así puedes contextualizar miles de chunks sin arruinarte.

[Anthropic blog · contextual retrieval](https://www.anthropic.com/engineering/contextual-retrieval)